# Natural-prevalence historical-window performance

Isolated analysis for `temporal_performance_windows.py`. Headline: **Death-F1@0.5**. Frozen-threshold F1 and death average precision are secondary; oracle F1 is diagnostic only. All inputs are checksum-validated through the canonical manifest pointer.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from temporal_performance_windows import load_window_experiment

sns.set_theme(style='whitegrid', context='talk', palette='colorblind')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.spines.top': False, 'axes.spines.right': False})
pointer = Path(os.environ.get('TEMPORAL_WINDOWS_MANIFEST', 'stats/temporal_performance_windows/latest_manifest.json'))
loaded = load_window_experiment(pointer)
manifest = loaded['manifest']
tables = {name: pd.DataFrame(rows) for name, rows in loaded['artifacts'].items()}
print(loaded['manifest_path'])
print('post-death exclusions:', manifest['post_death_exclusion_count'])

## Protocol and completeness checks

In [ ]:
assert manifest['complete'] is True
assert manifest['headline_metric'] == 'death_f1_at_0_5'
assert manifest['oracle_metrics_are_diagnostic_only'] is True
assert not manifest['failures'], manifest['failures']
assert tables['legacy_hard_label_parity']['parity'].all()
metrics = tables['yearly_metrics'].copy()
thresholds = tables['thresholds'].copy()
display(thresholds.groupby('window')[['training_record_count', 'training_death_count', 'threshold']].agg(['count', 'median', 'min', 'max']))
display(metrics.groupby(['window', 'cohort'])['valid'].agg(['count', 'sum']))

## Headline trajectories
Split results are averaged inside reference year before reference-year cluster-bootstrap inference.

In [ ]:
trajectory = tables['split_averaged_trajectories']
headline = trajectory[(trajectory.metric == 'death_f1_at_0_5') & (trajectory.cohort == 'all_comer')]
ax = sns.lineplot(data=headline, x='temporal_distance', y='split_mean', hue='window', marker='o')
ax.set(title='Death-F1@0.5 by historical window', xlabel='Years after reference', ylabel='Death-F1@0.5', ylim=(0, 1))
ax.legend(title='Window', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()
display(tables['trajectory_cluster_bootstrap'].query("cohort == 'all_comer'"))

## Threshold, calibration, and discrimination decomposition

In [ ]:
decomposition = metrics[(metrics.cohort == 'all_comer') & metrics.valid].melt(
    id_vars=['window', 'temporal_distance'],
    value_vars=['death_f1_at_0_5', 'death_f1_at_frozen_threshold', 'death_average_precision', 'death_f1_oracle'],
    var_name='measure', value_name='value')
g = sns.relplot(data=decomposition, x='temporal_distance', y='value', hue='measure', col='window', col_wrap=2, kind='line', marker='o', height=4)
g.set_axis_labels('Years after reference', 'Score').set(ylim=(0, 1))
g.fig.suptitle('Operational, frozen, discrimination, and diagnostic oracle measures', y=1.02)
plt.show()

## Paired window contrasts
Numeric differences exist only where record support matches the `reference_only_common` cell exactly.

In [ ]:
contrasts = tables['paired_window_contrasts']
matched = contrasts[(contrasts.matched_support) & (contrasts.metric == 'death_f1_at_0_5') & (contrasts.cohort == 'all_comer')]
ax = sns.lineplot(data=matched, x='temporal_distance', y='difference', hue='window', marker='o')
ax.axhline(0, color='black', linewidth=1)
ax.set(title='Paired Death-F1@0.5 difference vs reference-only common protocol', ylabel='Paired difference', xlabel='Years after reference')
plt.tight_layout(); plt.show()
display(tables['contrast_cluster_bootstrap'])

## Exposure cohorts and diagnostic classifications

In [ ]:
diagnostics = tables['diagnostic_classifications']
display(pd.crosstab([diagnostics.window, diagnostics.cohort], diagnostics.classification))
cohort_view = metrics[(metrics.window == 'all_history') & metrics.valid]
ax = sns.lineplot(data=cohort_view, x='temporal_distance', y='death_f1_at_0_5', hue='cohort', marker='o')
ax.set(title='All-history operational performance by exposure cohort', ylim=(0, 1))
plt.tight_layout(); plt.show()

## Interpretation guardrails

- Declining Death-F1@0.5 with stable average precision and oracle F1: threshold or probability-scaling failure.
- Stable average precision with worsening frozen F1, Brier score, or calibration: calibration, prevalence, or threshold drift.
- Declining average precision and oracle F1: discrimination decay.
- Poor average precision and every F1 variant at distance zero: inadequate baseline signal.

Oracle results remain diagnostic. Conclusions use matched support, natural prevalence, and split-within-reference aggregation.